# 06 · El módulo `random`

El módulo `random` de la biblioteca estándar implementa un generador de números **pseudoaleatorios** (Mersenne Twister). Las sucesiones que produce son deterministas, pero estadísticamente indistinguibles de una sucesión de variables uniformes independientes para los fines de este curso. Fijar la **semilla** hace reproducibles los resultados.

**Contenido**

1. Funciones básicas
2. Permutaciones aleatorias
3. Volados: probabilidad frecuentista
4. Volados: probabilidad clásica

In [1]:
import random
from collections import Counter
from itertools import product

random.seed(2026)

## 1. Funciones básicas

| función | resultado |
|---|---|
| `random.random()` | uniforme en $[0,1)$ |
| `random.uniform(a, b)` | uniforme en $[a,b]$ |
| `random.randint(a, b)` | entero uniforme en $\{a,\dots,b\}$ (incluye ambos extremos) |
| `random.randrange(start, stop, step)` | elemento uniforme de `range(start, stop, step)` |
| `random.choice(seq)` | elemento uniforme de una sucesión no vacía |
| `random.choices(seq, weights, k)` | $k$ elecciones con reemplazo y pesos |
| `random.sample(seq, k)` | $k$ elecciones sin reemplazo |

In [2]:
print(random.random())
print(random.uniform(0, 100))
print(random.randint(0, 10))
print([random.randrange(0, 100, 3) for _ in range(5)])   # múltiplos de 3 menores que 100

0.11911988496396309
50.251575523125055
8
[18, 42, 78, 93, 84]


`random.choice` acepta cualquier sucesión, con elementos de cualquier tipo.

In [3]:
print(random.choice(["rojo", "blanco", "verde"]))
print(random.choice([1, 2000, True, False, "Ray"]))
print(random.choices(["sol", "águila"], weights=[0.3, 0.7], k=10))
print(random.sample(range(1, 51), 6))   # un boleto de lotería: 6 números distintos

rojo
1
['águila', 'sol', 'águila', 'águila', 'águila', 'águila', 'águila', 'sol', 'sol', 'águila']
[23, 25, 48, 33, 41, 5]


## 2. Permutaciones aleatorias

`random.shuffle` baraja una lista **en su lugar** (modifica la lista y devuelve `None`); `random.sample(lista, len(lista))` devuelve una copia barajada. Cada una de las $n!$ permutaciones tiene la misma probabilidad.

In [4]:
baraja = list(range(10))
random.shuffle(baraja)
print(baraja)
print(random.sample(range(10), 10))

[9, 7, 0, 6, 8, 3, 2, 4, 1, 5]
[5, 7, 6, 0, 3, 4, 9, 8, 1, 2]


Para arreglos de NumPy se usa el generador de NumPy: `rng.permutation(arr)` devuelve una copia permutada y `rng.shuffle(arr)` permuta en su lugar.

In [5]:
import numpy as np

rng = np.random.default_rng(2026)
arr = np.array([1, 2, 3, 4, 5])
print(rng.permutation(arr), arr)

[4 2 5 1 3] [1 2 3 4 5]


**Verificación.** Al barajar $3$ cartas, las $3!=6$ permutaciones deben aparecer con frecuencia cercana a $1/6$.

In [6]:
n = 60_000
conteo = Counter(tuple(random.sample("abc", 3)) for _ in range(n))
{"".join(k): round(v / n, 4) for k, v in sorted(conteo.items())}

{'abc': 0.1708,
 'acb': 0.1668,
 'bac': 0.1643,
 'bca': 0.1671,
 'cab': 0.1646,
 'cba': 0.1664}

## 3. Volados: probabilidad frecuentista

Lanzamos una moneda justa $n$ veces y contamos cuántas veces cae cada cara. En un diccionario, a la izquierda están las llaves (*keys*) y a la derecha los valores (*values*).

In [7]:
resultados = {"águila": 0, "sol": 0}
caras = list(resultados)

for _ in range(10_000):
    resultados[random.choice(caras)] += 1

resultados

{'águila': 5068, 'sol': 4932}

Lo mismo codificando sol $=1$, águila $=0$:

In [8]:
omega = [random.randint(0, 1) for _ in range(10_000)]
print(f"soles = {omega.count(1)}, águilas = {omega.count(0)}, frecuencia de sol = {sum(omega) / len(omega):.4f}")

soles = 5032, águilas = 4968, frecuencia de sol = 0.5032


## 4. Volados: probabilidad clásica

Lanzamos $3$ veces una moneda justa de forma independiente. El espacio muestral es $\Omega=\{A,S\}^3$ y

$$\mathbb{P}(E)=\frac{\#E}{\#\Omega}.$$

In [9]:
Omega = set(product("AS", repeat=3))

def prob(E):
    return len(E) / len(Omega)

def prob_cond(E, F):
    """P(E | F) = P(E ∩ F) / P(F)."""
    return len(E & F) / len(F)

A = {w for w in Omega if w.count("A") == 3}   # tres águilas
B = {w for w in Omega if w[0] == "A"}         # águila en el primer lanzamiento
print("P(A) =", prob(A), "  P(A | B) =", prob_cond(A, B))

P(A) = 0.125   P(A | B) = 0.25


Saber que el primer lanzamiento fue águila cambia la probabilidad de tres águilas de $1/8$ a $1/4$.

**Ejercicio.** Estima $\mathbb{P}(A\mid B)$ por simulación: genera muchas ternas de volados, quédate con las que empiezan con águila y calcula la proporción de ternas con tres águilas.